# Run all Sahayak notebooks in one Colab runtime
Helper only, not a submission file. Everything lives in Google Drive, so a runtime restart loses nothing.

1. Upload the whole **`solution`** folder to `MyDrive/` (so you have `MyDrive/solution/learner/...`)
2. In Colab open the key icon (Secrets), add **`OPENAI_API_KEY`** and switch on Notebook access
3. Runtime: a normal CPU runtime is enough (the LLM is called through the API, no GPU needed)
4. Run the setup cell, then the notebook cells **one at a time, top to bottom**
5. Executed copies (with outputs) are saved to `MyDrive/solution/executed/` - those are the files to submit

**If the runtime restarts:** run the setup cell again, then continue from the notebook that was interrupted. Earlier results are already on Drive.

In [ ]:
import os, json, subprocess, sys
from google.colab import drive, userdata
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/solution'
LEARNER = f'{WORK_DIR}/learner'
OUT_DIR = f'{WORK_DIR}/executed'
NOTEBOOKS = [
    'adk_foundations.ipynb',
    'data_understanding_and_baseline.ipynb',
    'agent_pipeline_development.ipynb',
    'agent_evaluation_and_optimisation.ipynb',
]
missing = [n for n in NOTEBOOKS + ['sahayak_starter.py'] if not os.path.exists(f'{LEARNER}/{n}')]
assert not missing, f'Put these in MyDrive/solution/learner/ first: {missing}'
os.makedirs(OUT_DIR, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'google-adk', 'litellm', 'datasets', 'pandas',
                'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'nest_asyncio', 'pytest', 'sentence-transformers',
                'requests', 'fastapi', 'uvicorn'], check=True)
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

def run(nb):
    print(f'Running {nb} ... (no live output; this cell finishes when the notebook does)')
    r = subprocess.run(['jupyter', 'nbconvert', '--to', 'notebook', '--execute', '--allow-errors',
                        f'{LEARNER}/{nb}', '--output-dir', OUT_DIR, '--ExecutePreprocessor.timeout=-1',
                        '--ExecutePreprocessor.kernel_name=python3'], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f'{nb} failed - send the error above')
    cells = json.load(open(f'{OUT_DIR}/{nb}'))['cells']
    errs = [(i, o['ename'], o['evalue'][:120]) for i, c in enumerate(cells) if c['cell_type'] == 'code'
            for o in c.get('outputs', []) if o['output_type'] == 'error']
    print(f'Done -> {OUT_DIR}/{nb}  ({len(errs)} cell error(s))')
    for e in errs:
        print('   cell', *e)

assert os.environ.get('OPENAI_API_KEY'), 'Add OPENAI_API_KEY in Colab Secrets and switch on Notebook access'
print('Key loaded: True')
print('Already finished:', sorted(os.listdir(OUT_DIR)) or 'nothing yet')

In [ ]:
run('adk_foundations.ipynb')   # NB1

In [ ]:
run('data_understanding_and_baseline.ipynb')   # NB2

In [ ]:
run('agent_pipeline_development.ipynb')   # NB3

In [ ]:
run('agent_evaluation_and_optimisation.ipynb')   # NB4

## Optional: live dashboard (screenshots for the report)
Run after NB4. Starts `src/demo_app.py` and shows it inside Colab.

In [ ]:
import subprocess, sys, time
proc = subprocess.Popen([sys.executable, 'demo_app.py'], cwd=f'{WORK_DIR}/src',
                        env={**os.environ, 'PYTHONPATH': LEARNER})
time.sleep(20)
from google.colab import output
output.serve_kernel_port_as_iframe(7860, height=900)